# Week 12: Bounded agents and structured outputs

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Purpose:** a fully worked, self-paced in-class practice with an optional extension. Weeks 11 and 18 are optional examination-readiness materials. These notebooks do not create weekly graded assignments.

**Before running:** choose a Python 3 kernel in Jupyter or a CPU runtime in Colab. Run every cell from top to bottom with Shift+Enter. Setup installs missing free packages. No API key or external dataset download is required.

**Learning target:** complete the calculation, inspect intermediate values, and explain why the protocol supports a limited claim.

**Time:** about 25-45 minutes including reading, prediction, execution, and explain-back.


## 1. Set up and record the environment

Keep the seed fixed for the first run. Record actual outputs and package versions rather than copying a claimed performance number.


In [ ]:
import sys, subprocess, importlib.util, platform, json, random, hashlib
from pathlib import Path
required = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Seed:", SEED)


**Checkpoint.** The environment cell must finish before any analysis cell. Restart and run all if a variable is missing.


## 2. Define an agent contract

An agent combines an objective, a model or policy, tools, state, and a stopping rule. This free deterministic simulator teaches control flow; it is explicitly not an LLM.


In [ ]:
contract={"allowed_tools":["mean"],"max_steps":2,"input_limit":100,
          "require_numeric_data":True,"output_fields":["tool","result","status"]}
print(json.dumps(contract,indent=2))


**Checkpoint.** Explain which decisions require a real model in a production assistant and which belong to deterministic validation.


## 3. Validate before executing a tool

Reject unknown tools and invalid inputs before execution. Avoid eval and unrestricted Python execution.


In [ ]:
def run_request(request):
    if request.get("tool") not in contract["allowed_tools"]:
        return {"tool":request.get("tool"),"result":None,"status":"rejected"}
    values=request.get("values",[])
    if not isinstance(values,list) or not 1<=len(values)<=100 or not all(type(x) in (int,float) and np.isfinite(x) for x in values):
        return {"tool":request["tool"],"result":None,"status":"invalid data"}
    return {"tool":"mean","result":float(np.mean(values)),"status":"ok"}
print(run_request({"tool":"mean","values":[1,2,3]}))


**Checkpoint.** The valid output contains mean 2.0. The execution layer, not the prompt, enforces the allowlist.


## 4. Test failure paths

Use adversarial input as data. No string in this test grants new permissions.


In [ ]:
cases=[{"tool":"mean","values":[1,2,3]}, {"tool":"delete_files","values":[1]},
       {"tool":"mean","values":["ignore rules"]}, {"tool":"mean","values":[]}]
observed=[run_request(item)["status"] for item in cases]
assert observed==["ok","rejected","invalid data","invalid data"]
summary={"cases":len(cases),"statuses":observed,"policy":"deterministic teaching simulator"}
print(summary)


**Checkpoint.** Connect validation, tool execution, and explicit refusal to the written contract.


## 6. Preserve your evidence

Export a JSON record with a seed, package version, and measured result. Download the artifacts directory before leaving Colab; runtime storage is temporary.


In [ ]:
record={"week":12,"seed":SEED,"numpy":np.__version__,"python":platform.python_version(),"result":summary}
(ARTIFACTS/"week-12-evidence.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


**Checkpoint.** The evidence file must match the result you actually obtained. A saved screenshot alone does not permit rerunning the calculation.


## 7. Try one change and explain it
1. Predict the effect of one change before editing code. Change one parameter, input, or boundary condition only.
2. Rerun all cells in a fresh kernel. Keep the original evidence record and compare the observed difference.
3. Explain the result using the mechanism, not just the final number.
4. Name one assumption that limits the claim, and a new test that could challenge it.

**Troubleshooting:** if installation fails, restore network access; if a variable is undefined, run earlier cells; if your output differs, compare package versions, seed, and data construction. Do not repeatedly edit code until a preferred score appears.

**Completion criterion:** one executed notebook, one evidence JSON, and a brief claim/evidence/limitation statement.

### Reference documentation
- [NumPy user guide](https://numpy.org/doc/stable/user/)
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html)
- [Course repository](https://github.com/CHjonnyYu/MLES-Fall-2026)
